In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = os.path.join('..', 'data', 'fully combined')

qb_raw = pd.read_csv(os.path.join(DATA_DIR, 'qb_master.csv'))
rb_raw = pd.read_csv(os.path.join(DATA_DIR, 'rb_master.csv'))
te_raw = pd.read_csv(os.path.join(DATA_DIR, 'te_master.csv'))
wr_raw = pd.read_csv(os.path.join(DATA_DIR, 'wr_all_seasons_without_playoffs.csv'))

for name, df in [('QB', qb_raw), ('RB', rb_raw), ('TE', te_raw), ('WR', wr_raw)]:
    scol = 'Season' if 'Season' in df.columns else 'season'
    print(f'{name}: {df.shape[0]} redova x {df.shape[1]} kolona | Sezone: {df[scol].min()}-{df[scol].max()}')


QB: 845 redova x 188 kolona | Sezone: 1979-2025
RB: 622 redova x 115 kolona | Sezone: 1988-2025
TE: 351 redova x 70 kolona | Sezone: 1990-2025
WR: 5502 redova x 102 kolona | Sezone: 2015-2025


In [2]:
# QB: GS>0 filter (backup QB-ovi se uklanjaju - odluka iz EDA)
qb = qb_raw[qb_raw['GS'] > 0].copy()
qb['target'] = qb['Yds'] / qb['G'].replace(0, np.nan)

# RB: rushing yards per game
rb = rb_raw.copy()
rb['target'] = rb['Rush_Yds'] / rb['G'].replace(0, np.nan)

# TE: G>0 filter (igrači koji nisu odigrali ni jednu utakmicu — povreda, G=0 → target=NaN)
te = te_raw[te_raw['G'] > 0].copy()
te['target'] = te['Rec_Yds'] / te['G']

# WR: log transformacija na receiving_yards (odluka iz EDA - power-law distribucija)
# Clip na 0 pre log1p jer negativni yds/game su artefakti (penali)
wr = wr_raw.copy()
wr['rec_yds_per_game'] = wr['receiving_yards'] / wr['games_played'].replace(0, np.nan)
wr['target'] = np.log1p(wr['rec_yds_per_game'].clip(lower=0))

# Pregled
print('Ciljne promenljive:')
for name, df in [('QB', qb), ('RB', rb), ('TE', te)]:
    t = df['target']
    print(f'{name} (Yds/G):       mean={t.mean():.2f}, median={t.median():.2f}, min={t.min():.2f}, max={t.max():.2f}')

t_raw = wr['rec_yds_per_game']
t_log = wr['target']
print(f'WR (Yds/G raw):   mean={t_raw.mean():.2f}, median={t_raw.median():.2f}, min={t_raw.min():.2f}, max={t_raw.max():.2f}')
print(f'WR (log1p Yds/G): mean={t_log.mean():.3f}, median={t_log.median():.3f}, min={t_log.min():.3f}, max={t_log.max():.3f}')


Ciljne promenljive:
QB (Yds/G):       mean=223.72, median=231.05, min=0.00, max=371.20
RB (Yds/G):       mean=63.07, median=65.16, min=-0.23, max=131.06
TE (Yds/G):       mean=39.37, median=39.07, min=0.00, max=94.40
WR (Yds/G raw):   mean=22.39, median=16.57, min=-16.00, max=116.94
WR (log1p Yds/G): mean=2.737, median=2.866, min=0.000, max=4.770


In [3]:
# Award encoding — 5 binarnih kolona (odluka iz EDA)
# Kategorije: PB, AP-1, AP-2, MVP top5 (1-5), OPoY top5 (1-5)
# Isključujemo: Comeback Player of Year, rangiranja van top5

def encode_awards(series):
    def has(val, keywords):
        if pd.isna(val) or str(val).strip() == '':
            return 0
        return int(any(kw in str(val) for kw in keywords))
    
    df_enc = pd.DataFrame(index=series.index)
    df_enc['award_PB']       = series.apply(lambda v: has(v, ['PB']))
    df_enc['award_AP1']      = series.apply(lambda v: has(v, ['AP-1']))
    df_enc['award_AP2']      = series.apply(lambda v: has(v, ['AP-2']))
    df_enc['award_MVP_top5'] = series.apply(lambda v: has(v, ['MVP-1','MVP-2','MVP-3','MVP-4','MVP-5']))
    df_enc['award_OPoY_top5']= series.apply(lambda v: has(v, ['OPoY-1','OPoY-2','OPoY-3','OPoY-4','OPoY-5']))
    return df_enc

# QB i TE imaju 'Awards' (uppercase), RB ima 'awards' (lowercase)
qb = pd.concat([qb, encode_awards(qb['Awards'])], axis=1)
rb = pd.concat([rb, encode_awards(rb['awards'])], axis=1)
te = pd.concat([te, encode_awards(te['Awards'])], axis=1)
# WR nema awards kolonu — preskačemo

award_cols = ['award_PB','award_AP1','award_AP2','award_MVP_top5','award_OPoY_top5']
print('Award encoding — broj sezona po kategoriji:')
print(f'\n{"Nagrada":<20} {"QB":>6} {"RB":>6} {"TE":>6}')
print('-' * 38)
for col in award_cols:
    print(f'{col:<20} {qb[col].sum():>6} {rb[col].sum():>6} {te[col].sum():>6}')


Award encoding — broj sezona po kategoriji:

Nagrada                  QB     RB     TE
--------------------------------------
award_PB                262    152     93
award_AP1                40     52     28
award_AP2                37     39     20
award_MVP_top5           92     28      0
award_OPoY_top5          89     46      4


In [4]:
# ── Uklanjanje nepotrebnih kolona ─────────────────────────────────────────

# QB: metapodaci, defensive stats, elo (puno null-ova), special teams, duplikati
QB_DROP = (
    ['PlayerID', 'Pos', 'QBrec', 'Lng', 'TD%', 'Int%', 'Sk%', 'Y/G']  # metapodaci + duplikati
    + [c for c in qb.columns if c.startswith('def_')]                   # defensive stats
    + [c for c in qb.columns if c.startswith('elo_')]                   # elo kolone
    + ['snp_ST%', 'snp_special_teams']                                  # special teams snaps
)

# RB: metapodaci, defensive stats, special teams, duplikati kolona
RB_DROP = (
    ['PlayerID', 'Lg', 'Pos', 'Rush_Y/G', 'Rec_Y/G']                   # metapodaci + duplikati
    + [c for c in rb.columns if c.startswith('def_')]                   # defensive stats
    + ['snp_ST%', 'snp_ST_Snaps', 'snp_special_teams']                 # special teams
    + ['rec_success', 'catch_pct', 'Y/Tgt', 'Touch',                   # duplikati
       'yds_per_touch', 'yds_from_scrimmage', 'rush_receive_td', 'Rec_R/G']
)

# TE: metapodaci, special teams, defensive snaps
TE_DROP = (
    ['PlayerID', 'Pos']
    + ['snp_ST%', 'snp_ST_Snaps', 'snp_Def%', 'snp_Def_Snaps']
)

# WR: ID-evi, duplikati tima, quarter splits, win-prob splits, weather, game-state, betting
WR_DROP = (
    ['receiver_player_id', 'passer_player_id',                          # ID-evi
     'defteam', 'home_team', 'away_team', 'player_team']                # duplikati tima
    + [c for c in wr.columns if '_Q1' in c or '_Q2' in c
       or '_Q3' in c or '_Q4' in c]                                     # quarter splits
    + [c for c in wr.columns if c.startswith('yards_wp_')
       or c.startswith('receptions_wp_') or c.startswith('targets_wp_')]# win-prob splits
    + ['temp_f', 'humidity_pct', 'wind_mph',                            # weather
       'is_rain', 'is_snow', 'is_clear', 'is_dome', 'surface']         # stadium
    + ['avg_score_diff', 'avg_quarter', 'trailing_pct',                 # game-state
       'leading_pct', 'wp_var']
    + ['pregame_spread', 'pregame_total']                               # betting linije
)

# Primena — drop samo kolona koje postoje (ignorišemo ako neka već nedostaje)
qb.drop(columns=[c for c in QB_DROP if c in qb.columns], inplace=True)
rb.drop(columns=[c for c in RB_DROP if c in rb.columns], inplace=True)
te.drop(columns=[c for c in TE_DROP if c in te.columns], inplace=True)
wr.drop(columns=[c for c in WR_DROP if c in wr.columns], inplace=True)

# Statistika posle brisanja
print(f'{"Pozicija":<6} {"Pre":>6} {"Posle":>6} {"Izbačeno":>9}')
print('-' * 30)
for pos, raw, df in [('QB', qb_raw, qb), ('RB', rb_raw, rb), ('TE', te_raw, te), ('WR', wr_raw, wr)]:
    print(f'{pos:<6} {raw.shape[1]:>6} {df.shape[1]:>6} {raw.shape[1] - df.shape[1]:>9}')

print('\nPreostale kolone po poziciji:')
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{pos}: {list(df.columns)}')


Pozicija    Pre  Posle  Izbačeno
------------------------------
QB        188    123        65
RB        115     79        36
TE         70     70         0
WR        102     53        49

Preostale kolone po poziciji:

QB: ['Player', 'Season', 'Age', 'Team', 'G', 'GS', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Int', '1D', 'Succ%', 'Y/A', 'AY/A', 'Y/C', 'Rate', 'QBR', 'Sk', 'Yds_Lost', 'NY/A', 'ANY/A', '4QC', 'GWD', 'AV', 'Awards', 'adj_ANY/A+', 'adj_AY/A+', 'adj_Cmp%+', 'adj_Int%+', 'adj_NY/A+', 'adj_Rate+', 'adj_Sack%+', 'adj_TD%+', 'adj_Y/A+', 'adv_pass_pass_air_yds', 'adv_pass_pass_air_yds_per_att', 'adv_pass_pass_air_yds_per_cmp', 'adv_pass_pass_batted_passes', 'adv_pass_pass_blitzed', 'adv_pass_pass_drop_pct', 'adv_pass_pass_drops', 'adv_pass_pass_hits', 'adv_pass_pass_hurried', 'adv_pass_pass_on_target', 'adv_pass_pass_on_target_pct', 'adv_pass_pass_play_action', 'adv_pass_pass_play_action_pass_yds', 'adv_pass_pass_poor_throw_pct', 'adv_pass_pass_poor_throws', 'adv_pass_pass_pressured

In [5]:
# Analiza QB adv_ kolona: null% i korelacija sa osnovnim statistikama

adv_cols = [c for c in qb.columns if c.startswith('adv_')]
print(f'Ukupno adv_ kolona: {len(adv_cols)}\n')

# 1. Null procenat po koloni
null_pct = qb[adv_cols].isnull().mean() * 100
print('Null% po adv_ koloni:')
print(null_pct.sort_values(ascending=False).to_string())

# 2. Korelacija adv_ kolona sa osnovnim statistikama (Yds, Cmp, Att, TD, Rate itd.)
basic_stats = ['Yds', 'Cmp', 'Att', 'Cmp%', 'TD', 'Int', 'Rate', 'Sk', 'AV']
print('\n\nKorelacija adv_ kolona sa osnovnim statistikama (|r| > 0.85 = visoka duplikacija):')
print(f'{"adv kolona":<45} {"max |r|":>8}  {"najvise korelirana osnovna"}')
print('-' * 85)
for col in adv_cols:
    valid = qb[[col] + basic_stats].dropna()
    if len(valid) < 30:
        print(f'{col:<45} {"(premalo podataka)":>8}')
        continue
    corrs = valid[basic_stats].corrwith(valid[col]).abs()
    max_r = corrs.max()
    max_col = corrs.idxmax()
    marker = ' ◄ DUPLIKAT?' if max_r > 0.85 else ''
    print(f'{col:<45} {max_r:>8.3f}  {max_col}{marker}')


Ukupno adv_ kolona: 49

Null% po adv_ koloni:
adv_rr_rec_broken_tackles_per_rec      99.617347
adv_rr_rec_yac_per_rec                 95.408163
adv_rr_rec_air_yds_per_rec             95.408163
adv_rr_rec_drop_pct                    93.877551
adv_rr_rec_adot                        93.877551
adv_rr_rec_pass_rating                 93.877551
adv_rr_Rush_BrkTkl/A                   82.780612
adv_pass_pass_on_target_pct            71.045918
adv_pass_pass_rpo_yds                  70.918367
adv_pass_pass_rpo                      70.918367
adv_pass_pass_on_target                70.918367
adv_pass_pass_play_action              70.918367
adv_pass_pass_play_action_pass_yds     70.918367
adv_pass_pass_rpo_rush_yds             70.918367
adv_pass_pass_rpo_rush_att             70.918367
adv_pass_pass_rpo_pass_yds             70.918367
adv_pass_pass_rpo_pass_att             70.918367
adv_pass_rush_scrambles_yds_per_att    67.857143
adv_rr_Rec_AirYds                      67.346939
adv_rr_Rush_YBC        

In [6]:
# Izbacujemo sve adv_ kolone iz QB (67-100% null-ovi, duplikati osnovnih statistika)
adv_cols_to_drop = [c for c in qb.columns if c.startswith('adv_')]
qb.drop(columns=adv_cols_to_drop, inplace=True)

print(f'Izbačeno {len(adv_cols_to_drop)} adv_ kolona.')
print(f'QB shape: {qb.shape}')
print(f'Preostale kolone: {list(qb.columns)}')


Izbačeno 49 adv_ kolona.
QB shape: (784, 74)
Preostale kolone: ['Player', 'Season', 'Age', 'Team', 'G', 'GS', 'Cmp', 'Att', 'Cmp%', 'Yds', 'TD', 'Int', '1D', 'Succ%', 'Y/A', 'AY/A', 'Y/C', 'Rate', 'QBR', 'Sk', 'Yds_Lost', 'NY/A', 'ANY/A', '4QC', 'GWD', 'AV', 'Awards', 'adj_ANY/A+', 'adj_AY/A+', 'adj_Cmp%+', 'adj_Int%+', 'adj_NY/A+', 'adj_Rate+', 'adj_Sack%+', 'adj_TD%+', 'adj_Y/A+', 'rr_Fmb', 'rr_Rec', 'rr_Rec/G', 'rr_Rec_1D', 'rr_Rec_TD', 'rr_Rec_Y/G', 'rr_Rec_Yds', 'rr_Rush_1D', 'rr_Rush_A/G', 'rr_Rush_Att', 'rr_Rush_Lng', 'rr_Rush_Succ%', 'rr_Rush_TD', 'rr_Rush_Y/A', 'rr_Rush_Y/G', 'rr_Rush_Yds', 'rr_Tgt', 'rr_catch_pct', 'rr_rec_long', 'rr_rec_success', 'rr_rec_yds_per_rec', 'rr_rec_yds_per_tgt', 'rr_rush_receive_td', 'rr_touches', 'rr_yds_from_scrimmage', 'rr_yds_per_touch', 'snp_Def%', 'snp_Off%', 'snp_defense', 'snp_offense', 'Team_Changed', 'Prev_Season_Yds', 'target', 'award_PB', 'award_AP1', 'award_AP2', 'award_MVP_top5', 'award_OPoY_top5']


In [7]:
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{"="*60}')
    print(f'  {pos} — {df.shape[1]} kolona x {df.shape[0]} redova')
    print(f'{"="*60}')
    for i, col in enumerate(df.columns, 1):
        print(f'  {i:>3}. {col}')



  QB — 74 kolona x 784 redova
    1. Player
    2. Season
    3. Age
    4. Team
    5. G
    6. GS
    7. Cmp
    8. Att
    9. Cmp%
   10. Yds
   11. TD
   12. Int
   13. 1D
   14. Succ%
   15. Y/A
   16. AY/A
   17. Y/C
   18. Rate
   19. QBR
   20. Sk
   21. Yds_Lost
   22. NY/A
   23. ANY/A
   24. 4QC
   25. GWD
   26. AV
   27. Awards
   28. adj_ANY/A+
   29. adj_AY/A+
   30. adj_Cmp%+
   31. adj_Int%+
   32. adj_NY/A+
   33. adj_Rate+
   34. adj_Sack%+
   35. adj_TD%+
   36. adj_Y/A+
   37. rr_Fmb
   38. rr_Rec
   39. rr_Rec/G
   40. rr_Rec_1D
   41. rr_Rec_TD
   42. rr_Rec_Y/G
   43. rr_Rec_Yds
   44. rr_Rush_1D
   45. rr_Rush_A/G
   46. rr_Rush_Att
   47. rr_Rush_Lng
   48. rr_Rush_Succ%
   49. rr_Rush_TD
   50. rr_Rush_Y/A
   51. rr_Rush_Y/G
   52. rr_Rush_Yds
   53. rr_Tgt
   54. rr_catch_pct
   55. rr_rec_long
   56. rr_rec_success
   57. rr_rec_yds_per_rec
   58. rr_rec_yds_per_tgt
   59. rr_rush_receive_td
   60. rr_touches
   61. rr_yds_from_scrimmage
   62. rr_yds_per_

In [8]:
# QB: izbacujemo rr_* i snp_* kolone
qb_extra_drop = (
    [c for c in qb.columns if c.startswith('rr_')]
    + ['snp_Def%', 'snp_Off%', 'snp_defense', 'snp_offense']
)
qb.drop(columns=[c for c in qb_extra_drop if c in qb.columns], inplace=True)

# RB, TE, WR: izbacujemo adv_* i snp_* kolone
for df in [rb, te, wr]:
    drop = [c for c in df.columns if c.startswith('adv_') or c.startswith('snp_')]
    df.drop(columns=drop, inplace=True)

print(f'{"Pozicija":<6} {"Kolona":>7} {"Redova":>7}')
print('-' * 25)
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'{pos:<6} {df.shape[1]:>7} {df.shape[0]:>7}')


Pozicija  Kolona  Redova
-------------------------
QB          44     784
RB          41     622
TE          42     350
WR          53    5502


In [9]:
# Dodatno čišćenje kolona

# Awards raw string — više ne treba (već enkodovano u award_* binarnim kolonama)
for df, col in [(qb, 'Awards'), (rb, 'awards'), (te, 'Awards')]:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# QB: QBR — 33.9% null, strukturalni (postoji tek od ~2006), izbacujemo
if 'QBR' in qb.columns:
    qb.drop(columns=['QBR'], inplace=True)

# RB: receiving stats + kombinovane kolone sa null-ovima (rec komponenta nedostaje)
RB_DROP2 = ['Tgt', 'Rec', 'Rec_Yds', 'Rec_Y/R', 'Rec_TD', 'Rec_1D',
            'Rec_Succ%', 'Rec_Lng', 'Rec/G', 'Catch%', 'Rec_Y/Tgt', 'rec_long',
            'Y/Touch', 'Touches', 'Scrimmage_Yds', 'Rush_Rec_TD']
rb.drop(columns=[c for c in RB_DROP2 if c in rb.columns], inplace=True)

# TE: rushing stats (TE model predviđa receiving, rush stats su sekundarne)
TE_DROP2 = ['Rush_Att', 'Rush_Yds', 'Rush_TD', 'Rush_1D', 'Rush_Y/G', 'Rush_A/G',
            'Touches', 'Y/Touch', 'Scrimmage_Yds', 'Rush_Succ%', 'Rush_Lng', 'Rush_Y/A']
te.drop(columns=[c for c in TE_DROP2 if c in te.columns], inplace=True)

# WR: defensive deviation stats
WR_DROP2 = ['def_targets_dev', 'def_receptions_dev', 'def_yards_dev',
            'def_tds_dev', 'def_epa_dev']
wr.drop(columns=[c for c in WR_DROP2 if c in wr.columns], inplace=True)

print(f'{"Pozicija":<6} {"Kolona":>7} {"Redova":>7}')
print('-' * 25)
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'{pos:<6} {df.shape[1]:>7} {df.shape[0]:>7}')


Pozicija  Kolona  Redova
-------------------------
QB          42     784
RB          24     622
TE          29     350
WR          48    5502


In [10]:

for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    print(f'\n{"="*60}')
    print(f'  {pos} — {df.shape[1]} kolona x {df.shape[0]} redova')
    print(f'{"="*60}')
    for i, col in enumerate(df.columns, 1):
        print(f'  {i:>3}. {col}')



  QB — 42 kolona x 784 redova
    1. Player
    2. Season
    3. Age
    4. Team
    5. G
    6. GS
    7. Cmp
    8. Att
    9. Cmp%
   10. Yds
   11. TD
   12. Int
   13. 1D
   14. Succ%
   15. Y/A
   16. AY/A
   17. Y/C
   18. Rate
   19. Sk
   20. Yds_Lost
   21. NY/A
   22. ANY/A
   23. 4QC
   24. GWD
   25. AV
   26. adj_ANY/A+
   27. adj_AY/A+
   28. adj_Cmp%+
   29. adj_Int%+
   30. adj_NY/A+
   31. adj_Rate+
   32. adj_Sack%+
   33. adj_TD%+
   34. adj_Y/A+
   35. Team_Changed
   36. Prev_Season_Yds
   37. target
   38. award_PB
   39. award_AP1
   40. award_AP2
   41. award_MVP_top5
   42. award_OPoY_top5

  RB — 24 kolona x 622 redova
    1. Player
    2. Season
    3. Age
    4. Team
    5. G
    6. GS
    7. Rush_Att
    8. Rush_Yds
    9. Rush_TD
   10. Rush_1D
   11. Rush_Succ%
   12. Rush_Lng
   13. Rush_Y/A
   14. Rush_A/G
   15. Fmb
   16. av
   17. Team_Changed
   18. Prev_Season_Yds
   19. target
   20. award_PB
   21. award_AP1
   22. award_AP2
   23. award_MVP_to

In [11]:
# Analiza null vrednosti po poziciji
for pos, df in [('QB', qb), ('RB', rb), ('TE', te), ('WR', wr)]:
    null_counts = df.isnull().sum()
    has_nulls = null_counts[null_counts > 0]
    print(f'\n{"="*55}')
    print(f'  {pos} — kolone sa null vrednostima ({len(has_nulls)} od {df.shape[1]})')
    print(f'{"="*55}')
    if len(has_nulls) == 0:
        print('  Nema null vrednosti.')
    else:
        print(f'  {"Kolona":<35} {"Null#":>6}  {"Null%":>6}')
        print(f'  {"-"*52}')
        for col, cnt in has_nulls.sort_values(ascending=False).items():
            pct = cnt / len(df) * 100
            print(f'  {col:<35} {cnt:>6}  {pct:>5.1f}%')



  QB — kolone sa null vrednostima (11 od 42)
  Kolona                               Null#   Null%
  ----------------------------------------------------
  adj_ANY/A+                              18    2.3%
  adj_AY/A+                               18    2.3%
  adj_Cmp%+                               18    2.3%
  adj_Int%+                               18    2.3%
  adj_NY/A+                               18    2.3%
  adj_Rate+                               18    2.3%
  adj_Sack%+                              18    2.3%
  adj_TD%+                                18    2.3%
  adj_Y/A+                                18    2.3%
  Succ%                                    1    0.1%
  Y/C                                      1    0.1%

  RB — kolone sa null vrednostima (4 od 24)
  Kolona                               Null#   Null%
  ----------------------------------------------------
  Rush_Succ%                               3    0.5%
  Rush_Lng                                 3    0.5%
  Ru

In [12]:

# ── Kreiranje lag feature matrice (t-1 i t-2) ─────────────────────────────
# Model predviđa performansu sezone t koristeći SAMO podatke poznate prije sezone t:
#   - Sve statistike iz t-1 (_lag1) i t-2 (_lag2)
#   - Age i Team_Changed iz t (znamo ih prije sezone)
#   - target = Yds/G (ili log1p za WR) sezone t
# Nullovi za lag (rookies, 1. sezona u datasetu) → fill 0

POS_CONFIG = {
    'QB': dict(df=qb, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'RB': dict(df=rb, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'TE': dict(df=te, player_col='Player', season_col='Season',
               static_feats=['Age', 'Team_Changed'],
               id_cols=['Player', 'Season', 'Team']),
    'WR': dict(df=wr, player_col='receiver_player_name', season_col='season',
               static_feats=['age', 'team_changed'],
               id_cols=['receiver_player_name', 'season', 'posteam']),
}

lagged = {}
for pos, cfg in POS_CONFIG.items():
    df       = cfg['df'].copy()
    pcol     = cfg['player_col']
    scol     = cfg['season_col']
    id_cols  = cfg['id_cols']
    static   = [c for c in cfg['static_feats'] if c in df.columns]

    # Kolone koje lagujemo: sve osim id-eva, statičkih feature-a i targeta
    lag_cols = [c for c in df.columns
                if c not in id_cols + static + ['target']]

    # Baza: id + statički feature-i + target (sve iz sezone t)
    base = df[id_cols + static + ['target']].copy()

    # Helper DF samo za lag join
    lag_src = df[[pcol, scol] + lag_cols].copy()

    # Lag 1 — pomjeramo sezonu za +1 da matchamo sa tekućom sezonom t
    lag1 = lag_src.rename(columns={c: f'{c}_lag1' for c in lag_cols}).copy()
    lag1[scol] = lag1[scol] + 1
    base = base.merge(lag1, on=[pcol, scol], how='left')

    # Lag 2 — pomjeramo sezonu za +2
    lag2 = lag_src.rename(columns={c: f'{c}_lag2' for c in lag_cols}).copy()
    lag2[scol] = lag2[scol] + 2
    base = base.merge(lag2, on=[pcol, scol], how='left')

    # Nullovi (rookies / prva sezona u datasetu) → 0
    lag1_cols = [f'{c}_lag1' for c in lag_cols]
    lag2_cols = [f'{c}_lag2' for c in lag_cols]
    base[lag1_cols + lag2_cols] = base[lag1_cols + lag2_cols].fillna(0)

    # Ukloni redove bez targeta (edge cases G=0)
    base = base.dropna(subset=['target']).reset_index(drop=True)

    lagged[pos] = base

# Provjera
print(f'{"Pos":<5} {"Redova":>8} {"Kolona":>8} {"lag1 feats":>12} {"lag2 feats":>12}')
print('-' * 50)
for pos, df in lagged.items():
    lag1_cols = [c for c in df.columns if c.endswith('_lag1')]
    lag2_cols = [c for c in df.columns if c.endswith('_lag2')]
    print(f'{pos:<5} {df.shape[0]:>8} {df.shape[1]:>8} {len(lag1_cols):>12} {len(lag2_cols):>12}')


Pos     Redova   Kolona   lag1 feats   lag2 feats
--------------------------------------------------
QB         784       78           36           36
RB         622       42           18           18
TE         350       52           23           23
WR        6405       91           43           43


In [13]:

# Train/test split — temporalni: Season < 2024 = train, Season == 2024 = test
# Radi na lag feature matrici (lagged dict) — nema curenja podataka iz sezone t

POS_SPLIT = {
    'QB': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'RB': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'TE': dict(id_cols=['Player', 'Season', 'Team'],                    season_col='Season'),
    'WR': dict(id_cols=['receiver_player_name', 'season', 'posteam'],   season_col='season'),
}

splits = {}
for pos, cfg in POS_SPLIT.items():
    df       = lagged[pos]
    id_cols  = cfg['id_cols']
    scol     = cfg['season_col']

    train_df = df[df[scol] < 2024].copy()
    test_df  = df[df[scol] == 2024].copy()

    feat_cols = [c for c in df.columns if c not in id_cols + ['target']]

    X_train = train_df[feat_cols]
    y_train = train_df['target']
    X_test  = test_df[feat_cols]
    y_test  = test_df['target']

    splits[pos] = (X_train, X_test, y_train, y_test)

print(f'{"Pos":<5} {"X_train":>10} {"X_test":>8} {"y_train":>9} {"y_test":>8} {"features":>10}')
print('-' * 55)
for pos, (X_tr, X_te, y_tr, y_te) in splits.items():
    print(f'{pos:<5} {X_tr.shape[0]:>10} {X_te.shape[0]:>8} {y_tr.shape[0]:>9} {y_te.shape[0]:>8} {X_tr.shape[1]:>10}')


Pos      X_train   X_test   y_train   y_test   features
-------------------------------------------------------
QB           714       35       714       35         74
RB           552       34       552       34         38
TE           295       28       295       28         48
WR          5393      558      5393      558         87


In [14]:

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Binary kolone — ne skaliramo (već su 0/1, interpretabilnost se čuva)
# Award kolone sada imaju _lag1 / _lag2 sufiks (jer su lagovane)
# Team_Changed / team_changed ostaju statički (iz sezone t) → bez sufiksa
BINARY_COLS = [
    'award_PB_lag1',       'award_AP1_lag1',       'award_AP2_lag1',
    'award_MVP_top5_lag1', 'award_OPoY_top5_lag1',
    'award_PB_lag2',       'award_AP1_lag2',       'award_AP2_lag2',
    'award_MVP_top5_lag2', 'award_OPoY_top5_lag2',
    'Team_Changed', 'team_changed',
]

processed = {}

for pos, (X_train, X_test, y_train, y_test) in splits.items():

    # ── 1. Imputacija — fit samo na train ────────────────────────
    imputer = SimpleImputer(strategy='mean')
    X_train_imp = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_imp = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )

    # ── 2. Standardizacija — samo numeričke (ne binary) kolone ───
    binary_present = [c for c in BINARY_COLS if c in X_train.columns]
    scale_cols     = [c for c in X_train.columns if c not in binary_present]

    scaler = StandardScaler()
    X_train_imp[scale_cols] = scaler.fit_transform(X_train_imp[scale_cols])
    X_test_imp[scale_cols]  = scaler.transform(X_test_imp[scale_cols])

    processed[pos] = (X_train_imp, X_test_imp, y_train, y_test)

# Provjera — nullovi nakon imputacije
print(f'{"Pos":<5} {"X_train nulls":>14} {"X_test nulls":>13} {"scaled cols":>12} {"binary cols":>12}')
print('-' * 60)
for pos, (X_tr, X_te, _, _) in processed.items():
    binary_present = [c for c in BINARY_COLS if c in X_tr.columns]
    scale_cols     = [c for c in X_tr.columns if c not in binary_present]
    print(f'{pos:<5} {X_tr.isnull().sum().sum():>14} {X_te.isnull().sum().sum():>13} '
          f'{len(scale_cols):>12} {len(binary_present):>12}')


Pos    X_train nulls  X_test nulls  scaled cols  binary cols
------------------------------------------------------------
QB                 0             0           63           11
RB                 0             0           27           11
TE                 0             0           37           11
WR                 0             0           86            1


In [15]:

# Pregled kolona i null vrijednosti u processed datasetu

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    print(f'\n{"="*65}')
    print(f'  {pos} — X_train: {X_tr.shape[0]} redova x {X_tr.shape[1]} kolona')
    print(f'{"="*65}')
    
    # Sve kolone sa null statusom i tipom
    null_counts = X_tr.isnull().sum()
    print(f'\n  {"#":>4}  {"Kolona":<45} {"Tip":>8}  {"Nulls":>6}')
    print(f'  {"-"*70}')
    for i, col in enumerate(X_tr.columns, 1):
        null_n = null_counts[col]
        marker = ' ◄ NULL!' if null_n > 0 else ''
        print(f'  {i:>4}. {col:<45} {str(X_tr[col].dtype):>8}  {null_n:>6}{marker}')
    
    total_nulls = null_counts.sum()
    print(f'\n  Ukupno null-ova u X_train: {total_nulls}')
    print(f'  Ukupno null-ova u X_test:  {X_te.isnull().sum().sum()}')
    print(f'  y_train nulls: {y_tr.isnull().sum()} | y_test nulls: {y_te.isnull().sum()}')



  QB — X_train: 714 redova x 74 kolona

     #  Kolona                                             Tip   Nulls
  ----------------------------------------------------------------------
     1. Age                                            float64       0
     2. Team_Changed                                   float64       0
     3. G_lag1                                         float64       0
     4. GS_lag1                                        float64       0
     5. Cmp_lag1                                       float64       0
     6. Att_lag1                                       float64       0
     7. Cmp%_lag1                                      float64       0
     8. Yds_lag1                                       float64       0
     9. TD_lag1                                        float64       0
    10. Int_lag1                                       float64       0
    11. 1D_lag1                                        float64       0
    12. Succ%_lag1                

In [19]:

# ── Cross-Validation (TimeSeriesSplit, n=5) ────────────────────────────────
# Koristimo TimeSeriesSplit jer su podaci temporalni — svaki fold trenira na
# prošlosti i validira na budućnosti (nema data leakage).
# Modeli: LinearRegression, Ridge, Lasso, ElasticNet, KNeighbors, RandomForest
#
# NAPOMENA za WR: target je log1p(yds/g).
# Custom scoreri primjenjuju expm1 na predikcije i stvarne vrijednosti
# PRIJE računanja MAE/RMSE → sve metrike su u originalnoj yds/g skali.
# R² ostaje na log skali (konzistentno sa test evaluacijom).

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
import numpy as np, pandas as pd

MODELS = {
    'LinearRegression': LinearRegression(),
    'Ridge':            Ridge(alpha=1.0),
    'Lasso':            Lasso(alpha=0.01, max_iter=10000),
    'ElasticNet':       ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    'KNeighbors':       KNeighborsRegressor(n_neighbors=5),
    'RandomForest':     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
}

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae_inv(y_true, y_pred):
    """MAE na originalnoj yds/g skali (inverz log1p)."""
    return mean_absolute_error(np.expm1(y_true), np.expm1(y_pred))

def rmse_inv(y_true, y_pred):
    """RMSE na originalnoj yds/g skali (inverz log1p)."""
    return np.sqrt(mean_squared_error(np.expm1(y_true), np.expm1(y_pred)))

# Standardni scoreri (za QB, RB, TE)
scorers_std = {
    'MAE':  make_scorer(mean_absolute_error, greater_is_better=False),
    'RMSE': make_scorer(rmse,                greater_is_better=False),
    'R2':   make_scorer(r2_score,            greater_is_better=True),
}

# WR scoreri — MAE i RMSE na yds/g skali, R² na log skali
scorers_wr = {
    'MAE':  make_scorer(mae_inv,  greater_is_better=False),
    'RMSE': make_scorer(rmse_inv, greater_is_better=False),
    'R2':   make_scorer(r2_score, greater_is_better=True),
}

tscv = TimeSeriesSplit(n_splits=5)

cv_results = {}   # {pos: DataFrame sa CV metrikama}

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    rows = []
    scorers = scorers_wr if pos == 'WR' else scorers_std
    scale_note = ' [yds/g skala]' if pos == 'WR' else ''

    print(f'\n{"="*60}')
    print(f'  {pos} — TimeSeriesSplit CV (5 foldova, n_train={len(X_tr)}){scale_note}')
    print(f'{"="*60}')
    print(f'  {"Model":<20} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
    print(f'  {"-"*48}')

    for name, model in MODELS.items():
        cv = cross_validate(
            model, X_tr, y_tr,
            cv=tscv, scoring=scorers,
            return_train_score=False, n_jobs=-1
        )
        mae  = -cv['test_MAE'].mean()
        rmse_val = -cv['test_RMSE'].mean()
        r2   =  cv['test_R2'].mean()
        rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse_val, 'R2': r2})
        print(f'  {name:<20} {mae:>8.3f} {rmse_val:>8.3f} {r2:>8.3f}')

    cv_results[pos] = pd.DataFrame(rows).set_index('Model')

print('\nCV završen.')



  QB — TimeSeriesSplit CV (5 foldova, n_train=714)
  Model                     MAE     RMSE       R²
  ------------------------------------------------
  LinearRegression       45.275   68.693   -0.697
  Ridge                  36.954   50.006    0.172
  Lasso                  40.825   55.811   -0.048
  ElasticNet             36.862   49.752    0.176
  KNeighbors             38.957   52.409    0.095
  RandomForest           35.599   48.384    0.229

  RB — TimeSeriesSplit CV (5 foldova, n_train=552)
  Model                     MAE     RMSE       R²
  ------------------------------------------------
  LinearRegression       19.547   23.907    0.157
  Ridge                  18.811   22.962    0.233
  Lasso                  19.508   23.867    0.161
  ElasticNet             18.954   23.121    0.220
  KNeighbors             19.083   23.354    0.206
  RandomForest           18.105   21.969    0.299

  TE — TimeSeriesSplit CV (5 foldova, n_train=295)
  Model                     MAE     RMSE  

In [20]:

# ── Finalni trening na punom train setu + evaluacija na test setu (2024) ──
# Svaki model se trenira na cijelom X_train, evaluira na X_test (2024 sezona).
# Za WR: target je log1p(yds/g) → inverzno transformišemo za interpretabilne MAE/RMSE.

import copy

test_results = {}   # {pos: DataFrame sa test metrikama}
best_models  = {}   # {pos: (ime_modela, model_objekt)}

for pos, (X_tr, X_te, y_tr, y_te) in processed.items():
    rows = []
    trained = {}

    is_wr = (pos == 'WR')   # WR koristi log1p target

    print(f'\n{"="*60}')
    print(f'  {pos} — Test evaluacija (sezona 2024, n_test={len(X_te)})')
    print(f'{"="*60}')
    print(f'  {"Model":<20} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
    print(f'  {"-"*48}')

    for name, model_proto in MODELS.items():
        model = copy.deepcopy(model_proto)
        model.fit(X_tr, y_tr)
        preds = model.predict(X_te)

        if is_wr:
            # Inverz log1p za interpretabilne metrike (yds/g skala)
            y_true_inv = np.expm1(y_te)
            y_pred_inv = np.expm1(preds)
            mae  = mean_absolute_error(y_true_inv, y_pred_inv)
            rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
            r2   = r2_score(y_te, preds)   # R² ostaje na log skali (konzistentno sa CV)
        else:
            mae  = mean_absolute_error(y_te, preds)
            rmse = np.sqrt(mean_squared_error(y_te, preds))
            r2   = r2_score(y_te, preds)

        rows.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        trained[name] = model
        print(f'  {name:<20} {mae:>8.3f} {rmse:>8.3f} {r2:>8.3f}')

    df_test = pd.DataFrame(rows).set_index('Model')
    test_results[pos] = df_test

    # Odabir najboljeg modela — najmanji RMSE na test setu
    best_name = df_test['RMSE'].idxmin()
    best_models[pos] = (best_name, trained[best_name])
    print(f'\n  ★ Najbolji model ({pos}): {best_name} | RMSE={df_test.loc[best_name,"RMSE"]:.3f}')

print('\nFinalni trening završen.')



  QB — Test evaluacija (sezona 2024, n_test=35)
  Model                     MAE     RMSE       R²
  ------------------------------------------------
  LinearRegression       35.633   49.423   -0.208
  Ridge                  34.237   47.962   -0.138
  Lasso                  34.780   48.724   -0.174
  ElasticNet             34.285   47.385   -0.111
  KNeighbors             32.796   44.739    0.010
  RandomForest           32.256   42.528    0.105

  ★ Najbolji model (QB): RandomForest | RMSE=42.528

  RB — Test evaluacija (sezona 2024, n_test=34)
  Model                     MAE     RMSE       R²
  ------------------------------------------------
  LinearRegression       18.987   23.153    0.138
  Ridge                  19.064   23.173    0.137
  Lasso                  19.050   23.144    0.139
  ElasticNet             19.164   23.198    0.135
  KNeighbors             22.009   26.602   -0.137
  RandomForest           19.690   23.452    0.116

  ★ Najbolji model (RB): Lasso | RMSE=23.144



In [21]:

# ── Sumarni prikaz CV vs Test + čuvanje best modela ───────────────────────
import joblib, os

MODELS_DIR = os.path.join('..', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

print('CV vs Test — sumarni pregled (RMSE / R²)\n')
print(f'{"Pos":<5} {"Model":<20} {"CV_MAE":>8} {"CV_RMSE":>8} {"CV_R²":>7} │ {"Test_MAE":>9} {"Test_RMSE":>10} {"Test_R²":>8}')
print('─' * 90)

for pos in ['QB', 'RB', 'TE', 'WR']:
    cv_df   = cv_results[pos]
    test_df = test_results[pos]
    best    = best_models[pos][0]

    for i, model_name in enumerate(MODELS.keys()):
        marker = ' ★' if model_name == best else ''
        prefix = pos if i == 0 else ''
        cv_row   = cv_df.loc[model_name]
        test_row = test_df.loc[model_name]
        print(f'{prefix:<5} {model_name:<20} '
              f'{cv_row["MAE"]:>8.3f} {cv_row["RMSE"]:>8.3f} {cv_row["R2"]:>7.3f} │ '
              f'{test_row["MAE"]:>9.3f} {test_row["RMSE"]:>10.3f} {test_row["R2"]:>8.3f}{marker}')
    print('─' * 90)

# Čuvanje best modela po poziciji
print('\nSačuvani modeli:')
for pos, (name, model) in best_models.items():
    path = os.path.join(MODELS_DIR, f'{pos}_best_model.joblib')
    joblib.dump(model, path)
    print(f'  {pos}: {name} → {path}')


CV vs Test — sumarni pregled (RMSE / R²)

Pos   Model                  CV_MAE  CV_RMSE   CV_R² │  Test_MAE  Test_RMSE  Test_R²
──────────────────────────────────────────────────────────────────────────────────────────
QB    LinearRegression       45.275   68.693  -0.697 │    35.633     49.423   -0.208
      Ridge                  36.954   50.006   0.172 │    34.237     47.962   -0.138
      Lasso                  40.825   55.811  -0.048 │    34.780     48.724   -0.174
      ElasticNet             36.862   49.752   0.176 │    34.285     47.385   -0.111
      KNeighbors             38.957   52.409   0.095 │    32.796     44.739    0.010
      RandomForest           35.599   48.384   0.229 │    32.256     42.528    0.105 ★
──────────────────────────────────────────────────────────────────────────────────────────
RB    LinearRegression       19.547   23.907   0.157 │    18.987     23.153    0.138
      Ridge                  18.811   22.962   0.233 │    19.064     23.173    0.137
      Las